# Optimizer Comparison: Schedule-Free, Muon, Adafactor, and More

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/optimizers/optimizer_comparison.ipynb)

Optimizer choice is one of the most impactful but often overlooked hyperparameters in deep learning.
This notebook benchmarks five optimizers available in Ludwig on the UCI Wine Quality dataset:

| Optimizer | Key idea |
|---|---|
| **AdamW** (baseline) | Adam with decoupled weight decay — the standard choice |
| **RAdam** | Rectified Adam — warm-up-free, stable at the start of training |
| **Adafactor** | No second-moment buffer — large models, small memory footprint |
| **Schedule-Free AdamW** | No LR scheduler needed — the optimizer is the schedule |
| **Muon** | Newton-Schulz orthogonalisation of weight-matrix gradients |

The task is binary classification: predict whether a wine has quality >= 7.
Everything runs on CPU and completes in a few minutes.

## Setup

In [ ]:
!pip install ludwig scikit-learn --quiet

In [ ]:
import time
import tempfile
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from ludwig.api import LudwigModel

## Dataset

We use the **UCI Wine Quality** dataset (red wines).  
The target is binarised: `quality >= 7` → True, otherwise False.

In [ ]:
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/"
    "winequality-red.csv"
)

df = pd.read_csv(DATA_URL, sep=";")
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
df["quality"] = (df["quality"] >= 7).astype(str)  # binary: "True" / "False"

print(f"Shape: {df.shape}")
print(f"Class distribution:\n{df['quality'].value_counts()}")
df.head()

## Shared configuration

All models share the same architecture and 30 training epochs.
Only the `optimizer` block (and `learning_rate_scheduler`) differs.

In [ ]:
FEATURE_NAMES = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
    "pH", "sulphates", "alcohol",
]

INPUT_FEATURES = [{"name": n, "type": "number"} for n in FEATURE_NAMES]
OUTPUT_FEATURES = [{"name": "quality", "type": "binary"}]

BASE_TRAINER = {
    "epochs": 30,
}

# Storage for training curves and summary metrics
all_results = {}
summary = []

## Optimizer 1 — AdamW (baseline)

AdamW adds **decoupled weight decay** to Adam.  
It is the default optimizer in Ludwig and a solid baseline for most tasks.

```yaml
trainer:
  epochs: 30
  optimizer:
    type: adamw
    lr: 0.001
  learning_rate_scheduler:
    type: cosine
```

In [ ]:
config_adamw = {
    "model_type": "ecd",
    "input_features": INPUT_FEATURES,
    "output_features": OUTPUT_FEATURES,
    "trainer": {
        **BASE_TRAINER,
        "optimizer": {"type": "adamw", "lr": 0.001},
        "learning_rate_scheduler": {"type": "cosine"},
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    model = LudwigModel(config_adamw, logging_level=30)
    t0 = time.time()
    train_stats, _, _ = model.train(
        dataset=df,
        output_directory=tmpdir,
        skip_save_model=True,
        skip_save_progress=True,
        skip_save_log=True,
    )
    elapsed_adamw = time.time() - t0

all_results["adamw"] = train_stats.validation
summary.append({
    "optimizer": "adamw",
    "final_val_loss": train_stats.validation["quality"]["loss"][-1],
    "final_val_accuracy": train_stats.validation["quality"]["accuracy"][-1],
    "training_time_s": round(elapsed_adamw, 1),
})
print(f"AdamW done in {elapsed_adamw:.1f}s")

## Optimizer 2 — RAdam

**Rectified Adam** (Liu et al., 2019) automatically warms up the adaptive learning rate
by rectifying the variance of the second moment.  
Unlike AdamW, it does not need a manual warm-up schedule.

```yaml
trainer:
  epochs: 30
  optimizer:
    type: radam
    lr: 0.001
  learning_rate_scheduler:
    type: cosine
```

In [ ]:
config_radam = {
    "model_type": "ecd",
    "input_features": INPUT_FEATURES,
    "output_features": OUTPUT_FEATURES,
    "trainer": {
        **BASE_TRAINER,
        "optimizer": {"type": "radam", "lr": 0.001},
        "learning_rate_scheduler": {"type": "cosine"},
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    model = LudwigModel(config_radam, logging_level=30)
    t0 = time.time()
    train_stats, _, _ = model.train(
        dataset=df,
        output_directory=tmpdir,
        skip_save_model=True,
        skip_save_progress=True,
        skip_save_log=True,
    )
    elapsed_radam = time.time() - t0

all_results["radam"] = train_stats.validation
summary.append({
    "optimizer": "radam",
    "final_val_loss": train_stats.validation["quality"]["loss"][-1],
    "final_val_accuracy": train_stats.validation["quality"]["accuracy"][-1],
    "training_time_s": round(elapsed_radam, 1),
})
print(f"RAdam done in {elapsed_radam:.1f}s")

## Optimizer 3 — Adafactor

**Adafactor** (Shazeer & Stern, 2018) avoids storing a full second-moment matrix
by using a factored approximation.  
This makes it memory-efficient for large models (e.g., T5 was trained with Adafactor).
It includes internal step-size scaling, so an external LR scheduler is optional.

```yaml
trainer:
  epochs: 30
  optimizer:
    type: adafactor
    lr: 0.001
  # no learning_rate_scheduler needed
```

In [ ]:
config_adafactor = {
    "model_type": "ecd",
    "input_features": INPUT_FEATURES,
    "output_features": OUTPUT_FEATURES,
    "trainer": {
        **BASE_TRAINER,
        "optimizer": {"type": "adafactor", "lr": 0.001},
        # No learning_rate_scheduler — Adafactor handles step-size internally
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    model = LudwigModel(config_adafactor, logging_level=30)
    t0 = time.time()
    train_stats, _, _ = model.train(
        dataset=df,
        output_directory=tmpdir,
        skip_save_model=True,
        skip_save_progress=True,
        skip_save_log=True,
    )
    elapsed_adafactor = time.time() - t0

all_results["adafactor"] = train_stats.validation
summary.append({
    "optimizer": "adafactor",
    "final_val_loss": train_stats.validation["quality"]["loss"][-1],
    "final_val_accuracy": train_stats.validation["quality"]["accuracy"][-1],
    "training_time_s": round(elapsed_adafactor, 1),
})
print(f"Adafactor done in {elapsed_adafactor:.1f}s")

## Optimizer 4 — Schedule-Free AdamW

**Schedule-Free AdamW** (Defazio et al., 2024) eliminates the need for a separate
learning-rate schedule by building the schedule into the optimizer itself via
a momentum-based interpolation between two internal sequences.

> **Important:** do _not_ add a `learning_rate_scheduler` block when using this optimizer.
> Adding one would counteract the built-in schedule and harm convergence.

```yaml
trainer:
  epochs: 30
  optimizer:
    type: schedule_free_adamw
    lr: 0.001
  # No learning_rate_scheduler — this is the whole point!
```

In [ ]:
config_sfa = {
    "model_type": "ecd",
    "input_features": INPUT_FEATURES,
    "output_features": OUTPUT_FEATURES,
    "trainer": {
        **BASE_TRAINER,
        "optimizer": {"type": "schedule_free_adamw", "lr": 0.001},
        # No learning_rate_scheduler — Schedule-Free AdamW handles it internally
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    model = LudwigModel(config_sfa, logging_level=30)
    t0 = time.time()
    train_stats, _, _ = model.train(
        dataset=df,
        output_directory=tmpdir,
        skip_save_model=True,
        skip_save_progress=True,
        skip_save_log=True,
    )
    elapsed_sfa = time.time() - t0

all_results["schedule_free_adamw"] = train_stats.validation
summary.append({
    "optimizer": "schedule_free_adamw",
    "final_val_loss": train_stats.validation["quality"]["loss"][-1],
    "final_val_accuracy": train_stats.validation["quality"]["accuracy"][-1],
    "training_time_s": round(elapsed_sfa, 1),
})
print(f"Schedule-Free AdamW done in {elapsed_sfa:.1f}s")

## Optimizer 5 — Muon

**Muon** (Kosson et al.) applies Nesterov momentum updates orthogonalised via
Newton-Schulz iterations to weight matrices, while using AdamW for embeddings
and other non-matrix parameters.  
The orthogonalisation acts as a preconditioner with minimal overhead.

```yaml
trainer:
  epochs: 30
  optimizer:
    type: muon
    lr: 0.001
  learning_rate_scheduler:
    type: cosine
```

In [ ]:
config_muon = {
    "model_type": "ecd",
    "input_features": INPUT_FEATURES,
    "output_features": OUTPUT_FEATURES,
    "trainer": {
        **BASE_TRAINER,
        "optimizer": {"type": "muon", "lr": 0.001},
        "learning_rate_scheduler": {"type": "cosine"},
    },
}

with tempfile.TemporaryDirectory() as tmpdir:
    model = LudwigModel(config_muon, logging_level=30)
    t0 = time.time()
    train_stats, _, _ = model.train(
        dataset=df,
        output_directory=tmpdir,
        skip_save_model=True,
        skip_save_progress=True,
        skip_save_log=True,
    )
    elapsed_muon = time.time() - t0

all_results["muon"] = train_stats.validation
summary.append({
    "optimizer": "muon",
    "final_val_loss": train_stats.validation["quality"]["loss"][-1],
    "final_val_accuracy": train_stats.validation["quality"]["accuracy"][-1],
    "training_time_s": round(elapsed_muon, 1),
})
print(f"Muon done in {elapsed_muon:.1f}s")

## Results

In [ ]:
results_df = pd.DataFrame(summary)
results_df = results_df.set_index("optimizer")
results_df["final_val_loss"] = results_df["final_val_loss"].round(4)
results_df["final_val_accuracy"] = results_df["final_val_accuracy"].round(4)
print(results_df.to_string())

In [ ]:
COLORS = {
    "adamw": "#1f77b4",
    "radam": "#ff7f0e",
    "adafactor": "#2ca02c",
    "schedule_free_adamw": "#d62728",
    "muon": "#9467bd",
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Optimizer Comparison — Wine Quality (30 epochs, CPU)", fontsize=13)

for opt_name, val_stats in all_results.items():
    losses = val_stats["quality"]["loss"]
    accs = val_stats["quality"]["accuracy"]
    epochs = range(1, len(losses) + 1)
    color = COLORS.get(opt_name, None)
    axes[0].plot(epochs, losses, label=opt_name, color=color)
    axes[1].plot(epochs, accs, label=opt_name, color=color)

axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8)
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("optimizer_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved to optimizer_comparison.png")

## When to use each optimizer

| Optimizer | Best for | Watch out for |
|---|---|---|
| **AdamW** | General-purpose default; well-understood | Needs a good LR schedule and warm-up |
| **RAdam** | Short runs or when warm-up tuning is painful | Slightly more compute per step |
| **Adafactor** | Large models (LLMs, T5-scale); memory-constrained training | Can be slower to converge on small models |
| **Schedule-Free AdamW** | When you want to skip LR schedule tuning entirely | Must not add a `learning_rate_scheduler`; needs correct `warmup_steps` |
| **Muon** | Training deep networks with many weight matrices | Falls back to AdamW for embeddings/biases; newer, less battle-tested |
| **SOAP** | When you want Shampoo-quality second-order convergence with Adam memory | Higher per-step cost than Adam |

## Schedule-Free AdamW — important tip

> **Do not add `learning_rate_scheduler` when using `schedule_free_adamw`.**
>
> Schedule-Free AdamW internalises the learning-rate schedule via a
> momentum-based averaging trick.  Adding an external cosine or step-decay
> scheduler on top fights the internal schedule and typically hurts final
> accuracy.
>
> Correct config:
> ```yaml
> trainer:
>   optimizer:
>     type: schedule_free_adamw
>     lr: 0.001
> ```
>
> You may still set `warmup_steps` inside the optimizer block to give the
> internal schedule a short ramp-up.